# MathGlyph Pages Generation Examples

This notebook works in Google Colab from a fresh runtime. It installs `mathglyph-pages` when the repo source tree is not available, creates a tiny MathWriting-style fixture dataset, and then shows how to control page type, formula density, scan/photo distortions, and MathWriting split selection. For real generation, point `MATHWRITING_ROOT` to a mounted MathWriting 2024 dataset root.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import tempfile
import textwrap

from IPython.display import display

cwd = Path.cwd()
repo_root = None
for candidate in (cwd, cwd.parent):
    if (candidate / "src" / "mathglyph_pages").exists():
        repo_root = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if repo_root is None and importlib.util.find_spec("mathglyph_pages") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/reirei-00/mathglyph_pages.git",
    ])

from PIL import Image
from mathglyph_pages import MathPageConfig, generate_pages


def write_tiny_mathwriting(root: Path) -> Path:
    samples = {
        "train/linear.inkml": r'''
            <?xml version="1.0" encoding="UTF-8"?>
            <ink xmlns="http://www.w3.org/2003/InkML">
              <annotation type="sampleId">linear-1</annotation>
              <annotation type="normalizedLabel">y=x+1</annotation>
              <trace id="0">0 20, 15 5, 30 20, 45 5, 60 20</trace>
              <trace id="1">76 12, 106 12</trace>
              <trace id="2">126 4, 126 24, 145 24</trace>
            </ink>
        ''',
        "train/quadratic.inkml": r'''
            <?xml version="1.0" encoding="UTF-8"?>
            <ink xmlns="http://www.w3.org/2003/InkML">
              <annotation type="sampleId">quadratic-1</annotation>
              <annotation type="normalizedLabel">x^2+2x+1</annotation>
              <trace id="0">0 20, 14 4, 28 20</trace>
              <trace id="1">35 6, 42 1, 49 6</trace>
              <trace id="2">64 12, 95 12</trace>
              <trace id="3">112 20, 128 4, 144 20</trace>
              <trace id="4">160 12, 190 12</trace>
              <trace id="5">210 4, 210 24</trace>
            </ink>
        ''',
        "synthetic/matrix.inkml": r'''
            <?xml version="1.0" encoding="UTF-8"?>
            <ink xmlns="http://www.w3.org/2003/InkML">
              <annotation type="sampleId">matrix-1</annotation>
              <annotation type="normalizedLabel">\begin{bmatrix}1&amp;0\\0&amp;1\end{bmatrix}</annotation>
              <trace id="0">0 0, 0 70</trace>
              <trace id="1">8 10, 8 30</trace>
              <trace id="2">36 10, 52 10</trace>
              <trace id="3">8 48, 28 48</trace>
              <trace id="4">45 42, 45 62</trace>
              <trace id="5">64 0, 64 70</trace>
            </ink>
        ''',
    }
    for rel_path, text in samples.items():
        path = root / rel_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(textwrap.dedent(text).strip() + "\n", encoding="utf-8")
    return root

WORKDIR_HANDLE = tempfile.TemporaryDirectory()
WORKDIR = Path(WORKDIR_HANDLE.name)

if repo_root is not None and (repo_root / "tests" / "fixtures" / "tiny_mathwriting").exists():
    MATHWRITING_ROOT = repo_root / "tests" / "fixtures" / "tiny_mathwriting"
else:
    MATHWRITING_ROOT = write_tiny_mathwriting(WORKDIR / "tiny_mathwriting")

print("MathWriting root:", MATHWRITING_ROOT)
print("Temporary output dir:", WORKDIR)

In [ ]:
def run_example(name: str, **kwargs):
    config = MathPageConfig(
        mathwriting_root=MATHWRITING_ROOT,
        out_dir=WORKDIR / name,
        splits=("train", "synthetic"),
        max_scan_per_split=None,
        **kwargs,
    )
    result = generate_pages(config)
    summary = json.loads(result.summary_path.read_text(encoding="utf-8"))
    print(name, "pages=", summary["num_pages"], "regions=", summary["num_regions"])
    print("profiles:", summary["page_profiles"])
    print("styles:", summary["visual_styles"])
    if result.contact_sheet_path:
        display(Image.open(result.contact_sheet_path))
    return result, summary

## Clean sparse pages

Use `profile="formula_sparse"` for fewer formulas and `visual_style="clean"` for minimal augmentation.

In [ ]:
clean_sparse, clean_sparse_summary = run_example(
    "clean_sparse",
    num_pages=2,
    seed=101,
    profile="formula_sparse",
    visual_style="clean",
    formulas_per_page_min=4,
    formulas_per_page_max=5,
)

## Dense pages

Density is controlled by the page profile and the explicit formula-count bounds.

In [ ]:
dense, dense_summary = run_example(
    "dense",
    num_pages=2,
    seed=202,
    profile="formula_dense",
    visual_style="noisy_scan",
    formulas_per_page_min=12,
    formulas_per_page_max=14,
    formula_target_height_min=62,
    formula_target_height_max=88,
)

## Distortion controls

Choose a visual preset or override individual distortion fields directly.

In [ ]:
distorted, distorted_summary = run_example(
    "distorted",
    num_pages=2,
    seed=303,
    profile="formula_scattered",
    visual_style="clean",
    rotation_max_degrees=3.0,
    perspective_strength=0.05,
    yellow_strength=0.18,
    noise_strength=11,
    blur_radius=0.15,
    edge_shadow=True,
)

## Tables and matrix-heavy pages

`formula_matrix_table` requests matrix/array labels from MathWriting for part of the page.

In [ ]:
matrix_table, matrix_table_summary = run_example(
    "matrix_table",
    num_pages=1,
    seed=404,
    profile="formula_matrix_table",
    visual_style="aged_scan",
    formulas_per_page_min=6,
    formulas_per_page_max=8,
)

## Inspect metadata

Every formula region records which MathWriting split/file/sample drove it.

In [ ]:
rows = [json.loads(line) for line in matrix_table.metadata_path.read_text(encoding="utf-8").splitlines()]
formula_regions = [region for region in rows[0]["regions"] if region["type"] == "formula"]
formula_regions[:3]